# Denoising Method Comparison — POC_DDM

Compares **three** denoising methods on raw kinetic (LAMP/DDM) curves, each applied with
its current chosen hyperparameter (no HP sweep re-run in this notebook):

| # | Method | Hyperparameter |
|---|---|---|
| 1 | Moving avg (`ori_curves_avg`) | `config.WINDOW_SIZE_ORI` |
| 2 | Wavelet (universal threshold) | `WAVELETS[0]` = `'sym8'`, `LEVEL`, `THRESH_MODE` |
| 3 | Savitzky-Golay | `SG_POLYORDER`, `SG_OPTIMAL_W` (fixed below) |


In [1]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import joblib
import pywt
from scipy.signal import savgol_filter
from scipy.ndimage import uniform_filter1d

sys.path.insert(0, '/vol/bitbucket/gk225/POC_DDM/gk_code')
sys.path.insert(0, '/vol/bitbucket/gk225/POC_DDM/gk_code/main')
import config

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

print(f'pywt {pywt.__version__} | base: {config.BASE_FOLDER}')


[*] SUCCESS: TensorFlow is utilizing the GPU -> /physical_device:GPU:0



pywt 1.8.0 | base: /vol/bitbucket/gk225/POC_DDM_datasets


In [2]:
# ── Configuration ─────────────────────────────────────────────
# DATASET      = 'POC_DDM_final_nc_subtract'   # or 'POC_DDM_multi'
DATASET      = 'POC_DDM_final'
EXP_FOLDER   = os.path.join(config.BASE_FOLDER, DATASET)
CURVE_TYPE   = 'ori_curves'
WAVELETS     = ['sym8']          # first entry used in comparison
LEVEL        = None
THRESH_MODE  = 'soft'
SG_POLYORDER = 2                 # 2 = quadratic, 3 = cubic
SG_OPTIMAL_W = 69                # current chosen window -- derivative-test sweep on
                                  # D20260807_E00_C00_F4500KHz_U_DDM_02_07

In [3]:
# ── Wavelet ────────────────────────────────────────────────────────────────
def denoise_curve(curve, wavelet, level=LEVEL, mode=THRESH_MODE):
    coeffs     = pywt.wavedec(curve, wavelet, level=level)
    sigma      = np.median(np.abs(coeffs[-1])) / 0.6745
    threshold  = sigma * np.sqrt(2 * np.log(len(curve)))
    new_coeffs = [coeffs[0]] + [pywt.threshold(d, threshold, mode=mode) for d in coeffs[1:]]
    return pywt.waverec(new_coeffs, wavelet)[:len(curve)]

def denoise_all(curves, wavelet):
    it = tqdm(curves, desc=f'Denoising [{wavelet}]', unit='curve') if tqdm else curves
    return np.array([denoise_curve(c, wavelet) for c in it])

# ── SG ─────────────────────────────────────────────────────────────────────────────
def _ensure_odd(w): return w + (1 - w % 2)

def apply_sg(curves, window_length, polyorder):
    w = _ensure_odd(int(window_length)); w = max(w, polyorder + 2)
    return savgol_filter(curves, window_length=w, polyorder=polyorder, axis=1)

# ── Derivative-test sweep -- drives the SG/Wavelet HP search below ─────────────────
def _derivative_scores(curves, param_values, denoise_fn, sample_size=400, seed=0):
    rng    = np.random.default_rng(seed)
    sample = curves[rng.choice(len(curves), size=min(sample_size, len(curves)), replace=False)]
    _ref_w = max(5, _ensure_odd(int(sample.shape[1] * 0.03)))
    ref    = savgol_filter(sample, window_length=_ref_w, polyorder=2, axis=1)
    peak_r = np.abs(np.diff(ref, axis=1)).max(axis=1)
    ro_raw = np.std(np.diff(np.diff(sample, axis=1), axis=1), axis=1)
    prs, ros = [], []
    for p in param_values:
        df = np.diff(denoise_fn(sample, p), axis=1)
        prs.append(np.mean(np.abs(df).max(axis=1) / np.where(peak_r > 0, peak_r, 1)))
        ros.append(np.mean(np.std(np.diff(df, axis=1), axis=1) / np.where(ro_raw > 0, ro_raw, 1)))
    return np.array(prs), np.array(ros)

def _sweet_spot(param_values, roughnesses):
    """First (smallest) param where roughness drops within 2x of its floor (10th pctile)."""
    floor = np.percentile(roughnesses, 10)
    sweet = np.where(roughnesses <= 2.0 * floor)[0]
    return float(param_values[sweet[0]] if len(sweet) else param_values[np.argmin(roughnesses)])

# ── Data loading ────────────────────────────────────────────────────────────────
def list_folders(exp_folder):
    out = []
    for name in sorted(os.listdir(exp_folder)):
        p = os.path.join(exp_folder, name)
        if (os.path.isdir(p) and name not in config.EXCLUDED_FOLDERS
                and os.path.exists(os.path.join(p, config.TRAINING_DATA_PATH))):
            out.append((name, p))
    return out

def load_exp(folder_path):
    d           = joblib.load(os.path.join(folder_path, config.TRAINING_DATA_PATH))
    curves      = np.array(d['curves'][CURVE_TYPE])
    if 'ori_curves_avg' in d['curves']:
        curves_avg = np.array(d['curves']['ori_curves_avg'])
    else:
        curves_avg = uniform_filter1d(np.array(d['curves']['ori_curves']), size=config.WINDOW_SIZE_ORI, axis=1, mode='nearest')
    well_labels = np.array(d['well_labels'])
    return curves, curves_avg, well_labels

print('All functions loaded.')

All functions loaded.


In [4]:
import re
import pandas as pd
from scipy.stats import pearsonr

# ── Colours & display constants ───────────────────────────────────────────────
METHOD_COLORS  = ['#CC79A7', '#E69F00', '#0072B2']
SG_POLY_COLORS = {2: '#0072B2', 3: '#009E73', 4: '#D55E00'}
_M_COLOR       = {'raw': '#999999', 'smoothed': '#CC79A7', 'sg': '#0072B2'}
_WAVELET_COLORS_CYCLE = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#882255']

# HP search candidates (feed the SG/Wavelet HP search cells below)
SG_POLYORDERS      = [2, 3, 4]
WAVELET_CANDIDATES = [
    # 'db4', 'db6', 'db8',
    'sym4', 'sym6', 'sym8',
    # 'coif2', 'coif4',
    # 'bior3.5', 'bior4.4',
]

# ── Shared helpers ────────────────────────────────────────────────────────────
def _safe_corr(a, b):
    """Pearson r; returns np.nan if either input is constant."""
    return np.nan if np.std(a) == 0 or np.std(b) == 0 else pearsonr(a, b)[0]


def _full_metrics(raw, denoised):
    """Mean per-sample SNR, noise%, TV ratio, Pearson fidelity."""
    noise  = raw - denoised
    ns     = np.std(noise, axis=1)
    sr     = raw.max(axis=1) - raw.min(axis=1)
    vd, vn = np.var(denoised, axis=1), np.var(noise, axis=1)
    tv_d   = np.abs(np.diff(denoised, axis=1)).sum(axis=1)
    tv_r   = np.abs(np.diff(raw,      axis=1)).sum(axis=1)
    with np.errstate(divide='ignore', invalid='ignore'):
        snr_v   = np.where(vn > 0, 10*np.log10(np.where(vn > 0, vd/vn, 1.0)), np.nan)
        noise_v = np.where(sr > 0, ns / np.where(sr > 0, sr, 1.0) * 100, np.nan)
        tv_v    = tv_d / np.where(tv_r > 0, tv_r, np.nan)
    return {
        'snr':    float(np.nanmean(snr_v)),
        'noise%': float(np.nanmean(noise_v)),
        'tv':     float(np.nanmean(tv_v)),
        'corr':   float(np.nanmean([_safe_corr(raw[i], denoised[i]) for i in range(len(raw))])),
    }


# ── Method resolver ───────────────────────────────────────────────────────────
def _resolve_methods(r, methods):
    """Return [(array, title, color), ...] for the requested method keys.

    Keys:
      'smoothed'       moving average
      'sg'             SG (baseline polyorder + fixed window, see config)
      'sg_p2/3/4'      SG HP search result (per-polyorder auto window)
      'wv_<name>'      wavelet HP candidate  e.g. 'wv_sym6'
      '<name>'         baseline wavelet from WAVELETS  e.g. 'sym8'
    """
    wv_iter = iter(_WAVELET_COLORS_CYCLE)
    out = []
    for key in methods:
        if key == 'raw':
            out.append((r['raw'], 'Raw', _M_COLOR['raw']))
        elif key == 'smoothed':
            out.append((r['smoothed'],
                        f'Smoothed\n(w={config.WINDOW_SIZE_ORI})', _M_COLOR['smoothed']))
        elif key == 'sg':
            out.append((r.get('sg'),
                        f'SG p={SG_POLYORDER}\n(w={SG_OPTIMAL_W})', _M_COLOR['sg']))
        elif key.startswith('sg_p') and key[4:].isdigit():
            entry = r.get(key)
            if entry is not None:
                poly = int(key[4:])
                out.append((entry['denoised'],
                             f'SG p={poly}\n(w={entry["optimal_w"]})',
                             SG_POLY_COLORS.get(poly, '#888888')))
            else:
                print(f'[!] {key!r} not found — run SG HP search first')
        elif key.startswith('wv_'):
            wv_name = key[3:]
            entry   = r.get(key)
            if entry is not None:
                out.append((entry['denoised'],
                             f'Wavelet\n({wv_name})', next(wv_iter, '#888888')))
            elif wv_name in r.get('denoised', {}):
                out.append((r['denoised'][wv_name],
                             f'Wavelet\n({wv_name})', next(wv_iter, '#888888')))
            else:
                print(f'[!] {key!r} not found — run wavelet HP search first')
        elif key in r.get('denoised', {}):
            out.append((r['denoised'][key],
                        f'Wavelet\n({key})', next(wv_iter, '#888888')))
        else:
            print(f'[!] Unknown method key: {key!r}  (skipped)')
    return [(arr, lbl, col) for arr, lbl, col in out if arr is not None]


def _labels():
    return [
        f'Moving avg\n(w={config.WINDOW_SIZE_ORI})',
        f'Wavelet\n({WAVELETS[0]})',
        f'SG p={SG_POLYORDER}\n(w={SG_OPTIMAL_W})',
    ]

def _ready(r): return 'sg' in r


# ── Quantitative comparison ───────────────────────────────────────────────────
def compare_all_methods(folder_name, methods=None, plot=True):
    """
    Compute 7 denoising metrics and (optionally) plot bar charts.

    methods = None  → 3 baseline methods (smoothed, wavelet, sg), current hyperparams.
    methods = list  → any combination via _resolve_methods keys.

    Metrics: SNR, Noise%, Fidelity, AC lag-1, TV ratio, ΔTTP, SD_max ratio.
    """
    r = results[folder_name]
    if not _ready(r): print(f'[!] Run apply cell first for {folder_name}'); return
    raw = r['raw']

    if methods is None:
        labels  = _labels()
        arrays  = [r['smoothed'], r['denoised'][WAVELETS[0]], r['sg']]
        colors  = METHOD_COLORS
    else:
        resolved = _resolve_methods(r, methods)
        labels   = [l for _, l, _ in resolved]
        arrays   = [a for a, _, _ in resolved]
        colors   = [c for _, _, c in resolved]

    def tv(a): return np.abs(np.diff(a, axis=1)).sum(axis=1)
    def ac1(rw, dn):
        res = rw - dn
        return np.nanmean([np.corrcoef((e := res[i]-res[i].mean())[:-1], e[1:])[0, 1]
                           for i in range(len(res)) if res[i].std() > 0])

    mnames = ['SNR (dB)', 'Noise %', 'Fidelity\n(corr)', 'Residual\nAC lag-1',
              'TV ratio', 'Δ TTP', 'SD_max\nratio']
    better = ['↑', '↓', '↑', '↓', '↓', '↓', '→1']

    def best_idx(row, b):
        fin = np.isfinite(row)
        if not fin.any(): return None
        r_ = np.where(fin, row, np.nan)
        if b == '↓': return int(np.nanargmin(r_))
        if b == '↑': return int(np.nanargmax(r_))
        return int(np.nanargmin(np.abs(r_ - 1.0)))

    n_m  = len(labels)
    sc   = np.full((7, n_m), np.nan)
    _ref_w = max(5, _ensure_odd(int(raw.shape[1] * 0.03)))
    _ref   = savgol_filter(raw, window_length=_ref_w, polyorder=2, axis=1)
    draw   = np.abs(np.diff(_ref, axis=1)); ttp_raw = np.argmax(draw, axis=1).astype(float)
    for mi, den in enumerate(arrays):
        noise  = raw - den; ns = np.std(noise, axis=1); sr = raw.max(axis=1) - raw.min(axis=1)
        vd, vn = np.var(den, axis=1), np.var(noise, axis=1)
        dden   = np.abs(np.diff(den, axis=1))
        with np.errstate(divide='ignore', invalid='ignore'):
            sc[0,mi] = np.nanmean(np.where(vn > 0, 10*np.log10(np.where(vn > 0, vd/vn, 1.0)), np.nan))
            sc[1,mi] = np.nanmean(np.where(sr > 0, ns / np.where(sr > 0, sr, 1.0) * 100, np.nan))
        sc[2,mi] = np.nanmean([_safe_corr(raw[i], den[i]) for i in range(len(raw))])
        sc[3,mi] = ac1(raw, den)
        sc[4,mi] = np.nanmean(tv(den) / np.where(tv(raw) > 0, tv(raw), np.nan))
        sc[5,mi] = np.mean(np.abs(ttp_raw - np.argmax(dden, axis=1).astype(float)))
        sc[6,mi] = np.nanmean(dden.max(axis=1) / np.where(draw.max(axis=1)>0, draw.max(axis=1), np.nan))

    all_mn, all_sc, all_bt = mnames, list(sc), better

    if plot:
        fig, axes_pl = plt.subplots(1, len(all_mn), figsize=(max(6, 1.8*n_m), 5))
        if len(all_mn) == 1: axes_pl = [axes_pl]
        fig.suptitle(f'Quantitative Comparison — {folder_name}', fontsize=11, fontweight='bold')
        x = np.arange(n_m)
        for ax, metric, vals, b in zip(axes_pl, all_mn, all_sc, all_bt):
            best = best_idx(vals, b)
            for xi, (val, color) in enumerate(zip(vals, colors)):
                if np.isnan(val):
                    ax.bar(xi, 1, color='none', edgecolor=color, lw=1.5, ls='--', zorder=3)
                    ax.text(xi, 0.5, 'N/A', ha='center', va='center', fontsize=8, color=color, fontweight='bold')
                else:
                    bar = ax.bar(xi, val, color=color, edgecolor='black', zorder=3)
                    if xi == best: bar[0].set_edgecolor('red'); bar[0].set_linewidth(2.5)
                    ax.text(xi, val, f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)
            ax.set_title(metric, fontweight='bold', fontsize=9)
            ax.set_xticks(x)
            ax.set_xticklabels([l.replace('\n',' ') for l in labels], rotation=30, ha='right', fontsize=8)
            ax.grid(axis='y', alpha=0.3, zorder=0)
        axes_pl[0].set_ylabel('↑ higher = better', fontsize=8, color='gray')
        if len(axes_pl) > 3: axes_pl[3].set_ylabel('↓ lower = better', fontsize=8, color='gray')
        if len(axes_pl) > 6: axes_pl[6].set_ylabel('→ 1.0 = best',     fontsize=8, color='gray')
        note = '■ red outline = best  |  N/A = zero noise variance'
        fig.text(0.99, 0.01, note, ha='right', fontsize=7.5, color='red', style='italic')
        fig.tight_layout(); plt.show(); plt.close(fig)

    # Build DataFrame (methods as rows, metrics as columns)
    directions = {mn.replace('\n', ' '): bt for mn, bt in zip(all_mn, all_bt)}
    df = pd.DataFrame(
        {mn.replace('\n', ' '): [float(v) for v in vals]
         for mn, vals in zip(all_mn, all_sc)},
        index=[l.replace('\n', ' ') for l in labels],
    )

    def _style_col(col):
        d      = directions.get(col.name, '↑')
        finite = col.dropna()
        if finite.empty:
            return [''] * len(col)
        best_lbl = ((finite - 1.0).abs().idxmin() if d == '→1'
                    else finite.idxmin()           if d == '↓'
                    else finite.idxmax())
        return ['background-color: #c8f7c5; font-weight: bold'
                if i == best_lbl else '' for i in col.index]

    try:
        from IPython.display import display
        display(df.style.apply(_style_col).format('{:.4f}', na_rep='N/A')
                  .set_caption(folder_name))
    except Exception:
        print(df.round(4).to_string())

    return df


# ── Cross-chip averaging (shared by the "averaged metrics" cell and the LaTeX table) ──
def _method_family_key(label):
    """Groups a method label the same way regardless of chip -- needed because SG HP
    search's per-chip auto-tuned window means the SAME method ('SG p=2') carries a
    DIFFERENT window in its label on every chip ('SG p=2 (w=31)' vs '(w=27)', ...).
    Averaging by the raw label string would treat those as different methods; this
    strips just the w=.. part so they group together. Every other label's
    hyperparameter (Smoothed's w, Wavelet's mother function) is a fixed constant
    across chips already, so it needs no stripping."""
    m = re.match(r'^(SG p=\d+) \(w=\d+\)$', label)
    return m.group(1) if m else label


def _extract_sg_window(label):
    m = re.match(r'^SG p=\d+ \(w=(\d+)\)$', label)
    return int(m.group(1)) if m else None


def _average_metrics_df(metrics_by_folder, folders, metric_cols=None, show_window=True, return_std=False):
    """Method x metric DataFrame averaged across every folder, grouped by
    _method_family_key (not the raw label) so SG HP search's per-chip window doesn't
    split what's really the same method into separate rows. The displayed SG window
    is the mean of each chip's own optimal_w, rounded to the nearest odd integer
    (matching _ensure_odd) and prefixed with '~' since chips didn't all land on the
    same value -- e.g. 'SG p=2 (w=~29)'. Pass show_window=False to drop the
    window annotation entirely (e.g. for a plain method-name display). Pass
    return_std=True to also get the across-chip std, as a second DataFrame with
    the same (grouped, reindexed, relabeled) index -- for reporting spread
    alongside the mean (e.g. 'mean ± std') via the LaTeX average panel.

    metric_cols: which columns to average -- defaults to every column present in the
    per-folder DataFrames (all 7 metrics compare_all_methods computes); pass a subset
    (e.g. _LATEX_METRIC_COLS) to restrict it."""
    used = [metrics_by_folder[f[0]] for f in folders if f[0] in metrics_by_folder]
    if not used:
        raise ValueError('No folders with metrics to average.')
    cols = metric_cols if metric_cols is not None else list(used[0].columns)
    combined = pd.concat(used)
    family_keys = combined.index.map(_method_family_key)
    grouped = combined.groupby(family_keys)[cols]
    avg = grouped.mean()
    std = grouped.std() if return_std else None

    order, seen, sg_windows = [], set(), {}
    for lbl in used[0].index:
        fam = _method_family_key(lbl)
        if fam not in seen:
            seen.add(fam); order.append(fam)
    for df in used:
        for lbl in df.index:
            w = _extract_sg_window(lbl)
            if w is not None:
                sg_windows.setdefault(_method_family_key(lbl), []).append(w)

    def _display_label(fam):
        if fam in sg_windows and show_window:
            return f'{fam} (w=~{_ensure_odd(round(np.mean(sg_windows[fam])))})'
        return fam

    avg = avg.reindex(order)
    avg.index = [_display_label(f) for f in order]
    if return_std:
        std = std.reindex(order)
        std.index = avg.index
        return avg, std
    return avg


# ── Best-value highlighting (mirrors compare_all_methods' own per-column styling) ──
_METRIC_DIRECTIONS = {
    'SNR (dB)': '\u2191', 'Noise %': '\u2193', 'Fidelity (corr)': '\u2191',
    'Residual AC lag-1': '\u2193', 'TV ratio': '\u2193', '\u0394 TTP': '\u2193',
    'SD_max ratio': '\u21921', 'Curves/s': '\u2191', 'Peak Mem (MB)': '\u2193',
}


def _highlight_best(col, directions=_METRIC_DIRECTIONS):
    d = directions.get(col.name, '\u2191')
    finite = col.dropna()
    if finite.empty:
        return [''] * len(col)
    best_lbl = ((finite - 1.0).abs().idxmin() if d == '\u21921'
                else finite.idxmin()           if d == '\u2193'
                else finite.idxmax())
    return ['background-color: #c8f7c5; font-weight: bold'
            if i == best_lbl else '' for i in col.index]

print('All utility functions loaded.')

All utility functions loaded.


In [5]:
label_maps = config.get_label_mappings(EXP_FOLDER)
results    = {}
folders    = list_folders(EXP_FOLDER)[-6:]

print(f'{DATASET}: {len(folders)} folders')
for folder_name, folder_path in folders:
    print(f'\n─── {folder_name} ───')
    try:
        raw, smoothed, well_labels = load_exp(folder_path)
        print(f'  {raw.shape}')
        results[folder_name] = {
            'raw':         raw,
            'smoothed':    smoothed,
            'denoised':    {w: denoise_all(raw, wavelet=w) for w in WAVELETS},
            'well_labels': well_labels,
            'label_map':   label_maps.get(folder_name, {}),
        }
    except Exception as e:
        print(f'  [ERROR] {e}')

POC_DDM_final: 6 folders

─── D20260825_E00_C00_F4500KHz_U_DDM_05_01 ───


  (17350, 908)


Denoising [sym8]:   0%|          | 0/17350 [00:00<?, ?curve/s]

Denoising [sym8]:   3%|▎         | 473/17350 [00:00<00:03, 4720.12curve/s]

Denoising [sym8]:   5%|▌         | 946/17350 [00:00<00:03, 4625.47curve/s]

...

Denoising [sym8]:  90%|█████████ | 14884/16480 [00:03<00:00, 4292.56curve/s]

Denoising [sym8]:  93%|█████████▎| 15359/16480 [00:03<00:00, 4421.69curve/s]

Denoising [sym8]:  96%|█████████▌| 15804/16480 [00:03<00:00, 4406.41curve/s]

Denoising [sym8]:  99%|█████████▉| 16280/16480 [00:03<00:00, 4508.11curve/s]

Denoising [sym8]: 100%|██████████| 16480/16480 [00:03<00:00, 4571.88curve/s]

---
## Savitzky-Golay Smoothing

Fits a polynomial of degree $p$ to each sliding window of $w$ points. Unlike moving
average ($p=1$), it preserves peaks and the S-curve shape. `SG_OPTIMAL_W` above is the
current chosen window (fixed, not re-swept in this notebook).


---
## Hyperparameter Search (per chip)

SG (polyorder × window) and Wavelet (mother function) each use the same derivative-test
sweep (`_derivative_scores` / `_sweet_spot`) to auto-select their window/candidate per
chip — this is what feeds the `sg_p2/p3/p4` and `wv_sym4/sym6/sym8` rows in the table below.


In [6]:
# ── SG HP Search: polyorder × window sweep ───────────────────────────────────
# Saves results[folder]['sg_p{poly}'] — re-running skips cached entries.

for folder_name, _ in folders:
    if folder_name not in results:
        continue
    r   = results[folder_name]
    raw = r['raw']
    T   = raw.shape[1]

    sg_windows = np.unique(np.array(
        [_ensure_odd(int(w)) for w in np.linspace(5, max(7, int(T * 0.25)), 40)]
    ))
    sg_windows = sg_windows[sg_windows >= 5]

    print(f'\n{folder_name}  (T={T})')
    for poly in SG_POLYORDERS:
        key = f'sg_p{poly}'
        if key in r:
            d = r[key]
            print(f'  SG p={poly}: cached  optimal_w={d["optimal_w"]}  '
                  f'SNR={d["metrics"]["snr"]:.1f}dB')
            continue
        fn       = lambda c, w, p=poly: apply_sg(c, w, p)
        prs, ros = _derivative_scores(raw, sg_windows.astype(float), fn)
        opt_w    = int(_sweet_spot(sg_windows.astype(float), ros))
        den      = apply_sg(raw, opt_w, poly)
        m        = _full_metrics(raw, den)
        r[key]   = dict(windows=sg_windows, prs=prs, ros=ros,
                        optimal_w=opt_w, denoised=den, metrics=m)
        print(f'  SG p={poly}: optimal_w={opt_w}  SNR={m["snr"]:.1f}dB  '
              f'TV={m["tv"]:.3f}  corr={m["corr"]:.4f}')

print('\nDone. Keys: sg_p2, sg_p3, sg_p4')


D20260825_E00_C00_F4500KHz_U_DDM_05_01  (T=908)


  SG p=2: optimal_w=113  SNR=13.9dB  TV=0.028  corr=0.9741


  SG p=3: optimal_w=113  SNR=13.9dB  TV=0.029  corr=0.9743


  SG p=4: optimal_w=113  SNR=14.2dB  TV=0.035  corr=0.9757

D20260825_E00_C00_F4500KHz_U_DDM_06_02  (T=915)


  SG p=2: optimal_w=113  SNR=10.9dB  TV=0.026  corr=0.9394


...

  SG p=3: optimal_w=71  SNR=12.8dB  TV=0.043  corr=0.9573


  SG p=4: optimal_w=71  SNR=13.1dB  TV=0.052  corr=0.9599

D20260827_E00_C00_F4500KHz_U_DDM_04_final_final  (T=868)


  SG p=2: optimal_w=109  SNR=13.0dB  TV=0.029  corr=0.9709


  SG p=3: optimal_w=109  SNR=13.0dB  TV=0.030  corr=0.9712


  SG p=4: optimal_w=109  SNR=13.3dB  TV=0.036  corr=0.9726

Done. Keys: sg_p2, sg_p3, sg_p4


In [7]:
# ── Wavelet HP Search: mother function comparison ─────────────────────────────
# Saves results[folder]['wv_<name>'] — re-running skips cached entries.

for folder_name, _ in folders:
    if folder_name not in results:
        continue
    r   = results[folder_name]
    raw = r['raw']
    T   = raw.shape[1]
    print(f'\n{folder_name}  (T={T})')

    for wv in WAVELET_CANDIDATES:
        key = f'wv_{wv}'
        if key in r:
            print(f'  Wavelet {wv}: cached')
            continue
        if wv in r.get('denoised', {}):
            den    = r['denoised'][wv]
            r[key] = dict(denoised=den, metrics=_full_metrics(raw, den), is_ref=True)
            print(f'  Wavelet {wv}: (already in WAVELETS)  SNR={r[key]["metrics"]["snr"]:.1f}dB')
            continue
        try:
            if pywt.Wavelet(wv).dec_len > T:
                print(f'  Wavelet {wv}: skip (filter > signal length)')
                continue
            den    = denoise_all(raw, wv)
            r[key] = dict(denoised=den, metrics=_full_metrics(raw, den), is_ref=False)
            m      = r[key]['metrics']
            print(f'  Wavelet {wv}: SNR={m["snr"]:.1f}dB  TV={m["tv"]:.3f}  '
                  f'corr={m["corr"]:.4f}')
        except Exception as e:
            print(f'  Wavelet {wv}: ERROR — {e}')

print('\nDone. Keys: wv_<name> for each candidate.')


D20260825_E00_C00_F4500KHz_U_DDM_05_01  (T=908)


Denoising [sym4]:   0%|          | 0/17350 [00:00<?, ?curve/s]

Denoising [sym4]:   2%|▏         | 360/17350 [00:00<00:04, 3598.04curve/s]

Denoising [sym4]:   4%|▍         | 720/17350 [00:00<00:04, 3474.12curve/s]

Denoising [sym4]:   6%|▋         | 1118/17350 [00:00<00:04, 3700.57curve/s]

...

Denoising [sym6]:  97%|█████████▋| 15969/16480 [00:03<00:00, 4154.80curve/s]

Denoising [sym6]: 100%|█████████▉| 16412/16480 [00:03<00:00, 4234.55curve/s]

Denoising [sym6]: 100%|██████████| 16480/16480 [00:03<00:00, 4345.37curve/s]

  Wavelet sym6: SNR=13.0dB  TV=0.026  corr=0.9711


  Wavelet sym8: (already in WAVELETS)  SNR=13.3dB

Done. Keys: wv_<name> for each candidate.


In [8]:
# ── Apply SG to all datasets ──────────────────────────────────────
# Override the current chosen window here if needed:
# SG_OPTIMAL_W = 21

print(f'SG    : w={SG_OPTIMAL_W}, p={SG_POLYORDER}')

for folder_name, _ in folders:
    if folder_name not in results: continue
    raw = results[folder_name]['raw']
    print(f'{folder_name}  ({raw.shape[0]} curves) ...', end=' ', flush=True)
    results[folder_name]['sg'] = apply_sg(raw, SG_OPTIMAL_W, SG_POLYORDER)
    print('done.')

print('\nAll methods applied.')

SG    : w=69, p=2
D20260825_E00_C00_F4500KHz_U_DDM_05_01  (17350 curves) ... 

done.
D20260825_E00_C00_F4500KHz_U_DDM_06_02  (16971 curves) ... 

done.
D20260827_E00_C00_F4500KHz_U_DDM_01_final_final  (16381 curves) ... 

done.
D20260827_E00_C00_F4500KHz_U_DDM_02_final_final  (16638 curves) ... 

done.
D20260827_E00_C00_F4500KHz_U_DDM_03_final_final  (16754 curves) ... 

done.
D20260827_E00_C00_F4500KHz_U_DDM_04_final_final  (16480 curves) ... 

done.

All methods applied.


In [9]:
denoising_metrics_by_folder = {}
for folder_name, folder_path in folders:
    denoising_metrics = compare_all_methods(
        folder_name, plot=False,
        methods=['smoothed', 'sg_p2', 'sg_p3', 'sg_p4', 'wv_sym4', 'wv_sym6', 'wv_sym8'])
    denoising_metrics_by_folder[folder_name] = denoising_metrics

,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),13.8972,5.2390,0.9744,0.1937,0.0329,159.7548,0.4170
SG p=2 (w=113),13.8758,5.2644,0.9741,0.2005,0.0281,159.0874,0.3600
SG p=3 (w=113),13.9026,5.2486,0.9743,0.1960,0.0288,176.0893,0.3748
SG p=4 (w=113),14.1793,5.0965,0.9757,0.1481,0.0346,199.3688,0.4782
Wavelet (sym4),13.6662,5.3715,0.9731,0.2372,0.0233,182.3522,0.6139
Wavelet (sym6),13.9260,5.2374,0.9744,0.1971,0.0250,181.5893,0.6027
Wavelet (sym8),14.2349,5.0675,0.9760,0.1422,0.0307,179.4965,0.6257


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),10.8743,6.4493,0.9399,0.1743,0.0314,181.7193,0.4064
SG p=2 (w=113),10.8807,6.4738,0.9394,0.1794,0.0264,181.7451,0.3431
SG p=3 (w=113),10.9153,6.4541,0.9398,0.1747,0.0272,206.1333,0.3679
SG p=4 (w=113),11.1996,6.2831,0.9431,0.1299,0.0332,226.4433,0.4736
Wavelet (sym4),10.6247,6.6226,0.9363,0.2210,0.0211,197.4390,0.6042
Wavelet (sym6),10.9117,6.4576,0.9396,0.1802,0.0227,198.7892,0.5924
Wavelet (sym8),11.2734,6.2421,0.9438,0.1221,0.0292,199.3188,0.6212


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),14.9901,4.4124,0.9809,0.1817,0.0358,128.7747,0.4611
SG p=2 (w=109),14.9970,4.4218,0.9808,0.1843,0.0317,129.4527,0.4119
SG p=3 (w=109),15.0258,4.4019,0.9810,0.1770,0.0325,141.7386,0.4305
SG p=4 (w=109),15.2709,4.2903,0.9820,0.1347,0.0382,161.6537,0.5368
Wavelet (sym4),15.0203,4.4134,0.9809,0.1875,0.0286,157.9740,0.6960
Wavelet (sym6),15.0284,4.4086,0.9809,0.1855,0.0281,154.5456,0.6561
Wavelet (sym8),15.3204,4.2671,0.9821,0.1303,0.0337,156.0750,0.6892


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),16.7053,3.8200,0.9845,0.1955,0.0393,129.8036,0.4776
SG p=2 (w=113),16.7066,3.8246,0.9845,0.1967,0.0349,130.5331,0.4286
SG p=3 (w=113),16.7451,3.8037,0.9846,0.1880,0.0360,149.1157,0.4645
SG p=4 (w=113),16.9744,3.7131,0.9853,0.1487,0.0408,160.6108,0.5502
Wavelet (sym4),16.4926,3.9121,0.9838,0.2392,0.0315,160.3797,0.6754
Wavelet (sym6),16.7506,3.8078,0.9846,0.1956,0.0325,161.8723,0.6623
Wavelet (sym8),17.0482,3.6822,0.9856,0.1392,0.0373,157.2091,0.6822


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),12.4262,5.7467,0.9540,0.1689,0.0388,112.8453,0.3168
SG p=2 (w=71),12.7703,5.5677,0.9569,0.1141,0.0422,112.7218,0.3540
SG p=3 (w=71),12.7943,5.5464,0.9573,0.1081,0.0434,123.2845,0.3742
SG p=4 (w=71),13.0863,5.3895,0.9599,0.0556,0.0523,137.2087,0.4739
Wavelet (sym4),12.4543,5.7362,0.9540,0.1722,0.0328,116.6317,0.4834
Wavelet (sym6),12.8367,5.5309,0.9575,0.1090,0.0373,118.8887,0.5031
Wavelet (sym8),12.8368,5.5294,0.9575,0.1083,0.0370,118.4841,0.4943


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio
Smoothed (w=50),12.9589,5.4448,0.9710,0.2047,0.0334,172.5493,0.4291
SG p=2 (w=109),12.9894,5.4541,0.9709,0.2066,0.0294,173.6042,0.3803
SG p=3 (w=109),13.0159,5.4308,0.9712,0.2000,0.0300,185.8833,0.3953
SG p=4 (w=109),13.2668,5.2934,0.9726,0.1588,0.0359,207.6481,0.5028
Wavelet (sym4),13.0059,5.4404,0.9711,0.2081,0.0261,187.1377,0.6584
Wavelet (sym6),13.0203,5.4339,0.9711,0.2061,0.0255,191.4665,0.6321
Wavelet (sym8),13.3173,5.2657,0.9729,0.1542,0.0312,187.0434,0.6547


### Speed & resource benchmark -- curves/s and peak memory per method

In [10]:
import time
import tracemalloc

BENCH_SAMPLE_SIZE = 400   # matches _derivative_scores' own sample_size convention
BENCH_REPEATS = 3


def _denoise_all_notqdm(curves, wavelet):
    """Same as denoise_all, minus the tqdm progress bar -- its terminal writes would
    unfairly add overhead to the wavelet method's timing vs the vectorized methods."""
    return np.array([denoise_curve(c, wavelet) for c in curves])


def _benchmark_method(fn, curves, n_repeats=BENCH_REPEATS):
    """Mean wall-time throughput (curves/s) and peak traced memory (MB) for a
    denoising call fn(curves) -> denoised, over n_repeats runs."""
    n = len(curves)
    times, peak_mb = [], 0.0
    for _ in range(n_repeats):
        tracemalloc.start()
        t0 = time.perf_counter()
        fn(curves)
        times.append(time.perf_counter() - t0)
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        peak_mb = max(peak_mb, peak / 1e6)
    mean_time = float(np.mean(times))
    return (n / mean_time if mean_time > 0 else np.nan), peak_mb


for folder_name, _ in folders:
    if folder_name not in results or folder_name not in denoising_metrics_by_folder:
        continue
    r      = results[folder_name]
    raw    = r['raw']
    rng    = np.random.default_rng(0)
    sample = raw[rng.choice(len(raw), size=min(BENCH_SAMPLE_SIZE, len(raw)), replace=False)]

    method_fns = {
        f'Smoothed (w={config.WINDOW_SIZE_ORI})':
            lambda c: uniform_filter1d(c, size=config.WINDOW_SIZE_ORI, axis=1, mode='nearest'),
    }
    for poly in SG_POLYORDERS:
        key = f'sg_p{poly}'
        if key in r:
            w = r[key]['optimal_w']
            method_fns[f'SG p={poly} (w={w})'] = lambda c, w=w, p=poly: apply_sg(c, w, p)
    wv_names = dict.fromkeys(list(WAVELETS) + [k[3:] for k in r if k.startswith('wv_')])
    for wv in wv_names:
        method_fns[f'Wavelet ({wv})'] = lambda c, wv=wv: _denoise_all_notqdm(c, wavelet=wv)

    print(f'{folder_name}: benchmarking {len(method_fns)} methods on {len(sample)} curves '
          f'({BENCH_REPEATS} reps each)...')
    speed_col, mem_col = {}, {}
    for label, fn in method_fns.items():
        cps, mem = _benchmark_method(fn, sample)
        speed_col[label] = cps
        mem_col[label]   = mem
        print(f'  {label}: {cps:8.1f} curves/s   {mem:6.2f} MB peak')

    df = denoising_metrics_by_folder[folder_name]
    df['Curves/s']      = pd.Series(speed_col)
    df['Peak Mem (MB)'] = pd.Series(mem_col)

print('\nSpeed/memory benchmark done.')

D20260825_E00_C00_F4500KHz_U_DDM_05_01: benchmarking 7 methods on 400 curves (3 reps each)...
  Smoothed (w=50): 265728.3 curves/s     2.91 MB peak
  SG p=2 (w=113):  15584.2 curves/s     3.70 MB peak
  SG p=3 (w=113):  14763.8 curves/s     3.69 MB peak


  SG p=4 (w=113):  14731.7 curves/s     3.69 MB peak


  Wavelet (sym8):   1388.4 curves/s     5.92 MB peak


  Wavelet (sym4):   1129.6 curves/s     5.92 MB peak


  Wavelet (sym6):   1228.7 curves/s     5.92 MB peak
D20260825_E00_C00_F4500KHz_U_DDM_06_02: benchmarking 7 methods on 400 curves (3 reps each)...
  Smoothed (w=50): 332330.4 curves/s     2.93 MB peak
  SG p=2 (w=113):  15759.7 curves/s     3.72 MB peak
  SG p=3 (w=113):  14277.4 curves/s     3.72 MB peak


...

  Wavelet (sym6):   1428.9 curves/s     3.69 MB peak
D20260827_E00_C00_F4500KHz_U_DDM_04_final_final: benchmarking 7 methods on 400 curves (3 reps each)...
  Smoothed (w=50): 446638.6 curves/s     2.78 MB peak
  SG p=2 (w=109):  17039.7 curves/s     3.54 MB peak
  SG p=3 (w=109):  16038.4 curves/s     3.54 MB peak


  SG p=4 (w=109):  15796.9 curves/s     3.54 MB peak


  Wavelet (sym8):   1355.0 curves/s     5.66 MB peak


  Wavelet (sym4):   1296.4 curves/s     5.66 MB peak


  Wavelet (sym6):   1267.2 curves/s     5.66 MB peak

Speed/memory benchmark done.


### Averaged metrics across all chips

All 7 metrics from `compare_all_methods`, averaged across every chip. SG rows are grouped
by polyorder only (`SG p=2`/`p=3`/`p=4`) -- the per-chip auto-tuned window is stripped
before averaging (`_method_family_key`) since it differs across chips; the displayed window
is the mean of each chip's own value, marked `w=~..`.


In [11]:
avg_denoising_metrics = _average_metrics_df(denoising_metrics_by_folder, folders, show_window=False)
print(f"Averaged across {len(folders)} chips:")
try:
    from IPython.display import display
    display(avg_denoising_metrics.style.apply(_highlight_best).format('{:.4f}', na_rep='N/A'))
except Exception:
    print(avg_denoising_metrics.round(4).to_string())


Averaged across 6 chips:


,SNR (dB),Noise %,Fidelity (corr),Residual AC lag-1,TV ratio,Δ TTP,SD_max ratio,Curves/s,Peak Mem (MB)
Smoothed (w=50),13.6420,5.1854,0.9675,0.1865,0.0353,147.5745,0.4180,416353.3635,2.6803
SG p=2,13.7033,5.1677,0.9678,0.1803,0.0321,147.8574,0.3797,16841.9285,3.4424
SG p=3,13.7332,5.1476,0.9680,0.1740,0.0330,163.7075,0.4012,17407.8721,3.4390
SG p=4,13.9962,5.0110,0.9698,0.1293,0.0392,182.1556,0.5026,15965.5031,3.4367
Wavelet (sym4),13.5440,5.2493,0.9665,0.2109,0.0272,166.9857,0.6219,1210.2726,5.4690
Wavelet (sym6),13.7456,5.1460,0.9680,0.1789,0.0285,167.8586,0.6081,1289.8372,5.4688
Wavelet (sym8),14.0052,5.0090,0.9697,0.1327,0.0332,166.2711,0.6279,1356.7512,5.4689


### LaTeX table -- Residual AC lag-1 / SNR (dB) / Fidelity (corr) per chip

One sub-table per chip (`DDM_0x` -> `Chip 0x`), restricted to the three metrics
above. Methods are grouped as `Smoothed: Simple Moving Average` / `SG:
Savitzky-Golay` / `Wavelet: DWT`, with the per-row hyperparameter
(`w=`/`p=.. w=..`/`sym..`) split into its own column.

In [12]:
import re
import string

def _chip_label(folder_name):
    m = re.search(r'DDM_(\d+)', folder_name)
    return f'Chip {int(m.group(1)):02d}' if m else folder_name


def _split_method_label(label):
    """'Smoothed (w=15)' -> ('Smoothed: Simple Moving Average', '(w=15)')
       'SG p=2 (w=31)'   -> ('SG: Savitzky-Golay', '(p=2 w=31)')
       'SG p=2 (w=~29)'  -> ('SG: Savitzky-Golay', '(p=2 w=~29)')  -- averaged-panel window
       'Wavelet (sym4)'  -> ('Wavelet: DWT', '(sym4)')"""
    m = re.match(r'^SG p=(\d+) \(w=(~?\d+)\)$', label)
    if m:
        return 'SG: Savitzky-Golay', f'(p={m.group(1)} w={m.group(2)})'
    m = re.match(r'^Smoothed \((w=\d+)\)$', label)
    if m:
        return 'Smoothed: Simple Moving Average', f'({m.group(1)})'
    m = re.match(r'^Wavelet \((.+)\)$', label)
    if m:
        return 'Wavelet: DWT', f'({m.group(1)})'
    return label, ''


_LATEX_METRIC_COLS      = ['Residual AC lag-1', 'SNR (dB)', 'Fidelity (corr)', 'Curves/s', 'Peak Mem (MB)']
_LATEX_METRIC_HEADERS   = ['Residual Autocorrelation (Lag-1)', 'SNR (dB)', 'Fidelity (Corr)', 'Curves/s', 'Peak Mem (MB)']
_LATEX_METRIC_FMT       = ['{:.3f}', '{:.2f}', '{:.3f}', '{:.0f}', '{:.2f}']
_LATEX_METRIC_DIRECTION = ['min', 'max', 'max', 'max', 'min']  # residual AC: lower is better; SNR/fidelity/speed: higher; memory: lower


def _latex_panel(df, panel_letter, chip_name, std_df=None):
    best_row = {
        col: (df[col].idxmin() if direction == 'min' else df[col].idxmax())
        for col, direction in zip(_LATEX_METRIC_COLS, _LATEX_METRIC_DIRECTION)
    }

    lines = [
        f'    ({panel_letter}) Performance on {chip_name}\\\\[0.5em]',
        '    \\resizebox{\\textwidth}{!}{',
        '    \\begin{tabular}{llrrr}',
        '    \\toprule',
        '    \\textbf{Method} & \\textbf{Hyperparameter} & \\textbf{'
        + '} & \\textbf{'.join(_LATEX_METRIC_HEADERS) + '} \\\\',
        '    \\midrule',
    ]
    for method_label, row in df.iterrows():
        method_name, hp = _split_method_label(method_label)
        cells = []
        for col, fmt in zip(_LATEX_METRIC_COLS, _LATEX_METRIC_FMT):
            cell = fmt.format(row[col])
            if std_df is not None and method_label in std_df.index and pd.notna(std_df.loc[method_label, col]):
                cell = f'{cell} $\\pm$ {fmt.format(std_df.loc[method_label, col])}'
            if method_label == best_row[col]:
                cell = f'\\textbf{{{cell}}}'
            cells.append(cell)
        lines.append(f'    {method_name} & {hp} & {" & ".join(cells)} \\\\')
    lines += ['    \\bottomrule', '    \\end{tabular}', '    }']
    return '\n'.join(lines)


def build_denoising_latex_table(metrics_by_folder, folders, caption, label, include_average=True):
    used_folders = [f for f in folders if f[0] in metrics_by_folder]
    panels = [
        _latex_panel(metrics_by_folder[folder_name], string.ascii_lowercase[i], _chip_label(folder_name))
        for i, (folder_name, _) in enumerate(used_folders)
    ]
    if include_average and used_folders:
        avg_df, std_df = _average_metrics_df(metrics_by_folder, used_folders,
                                              metric_cols=_LATEX_METRIC_COLS, return_std=True)
        panels.append(_latex_panel(avg_df, string.ascii_lowercase[len(used_folders)],
                                   f'Mean of All {len(used_folders)} Chips', std_df=std_df))
    body = '\n\n    \\vspace{2.5em}\n\n'.join(panels)
    return (
        '\\begin{table}[htbp]\n'
        '    \\centering\n'
        f'    \\caption{{{caption}}}\n'
        f'    \\label{{{label}}}\n'
        '    \\small\n\n'
        f'{body}\n'
        '\\end{table}'
    )


denoising_latex = build_denoising_latex_table(
    denoising_metrics_by_folder, folders,
    caption='Denoising method comparison across the evaluated chips.',
    label='tab:denoising_comparison',
)
print(denoising_latex)

\begin{table}[htbp]
    \centering
    \caption{Denoising method comparison across the evaluated chips.}
    \label{tab:denoising_comparison}
    \small

    (a) Performance on Chip 05\\[0.5em]
    \resizebox{\textwidth}{!}{
    \begin{tabular}{llrrr}
    \toprule
    \textbf{Method} & \textbf{Hyperparameter} & \textbf{Residual Autocorrelation (Lag-1)} & \textbf{SNR (dB)} & \textbf{Fidelity (Corr)} & \textbf{Curves/s} & \textbf{Peak Mem (MB)} \\
    \midrule
    Smoothed: Simple Moving Average & (w=50) & 0.194 & 13.90 & 0.974 & \textbf{265728} & \textbf{2.91} \\
    SG: Savitzky-Golay & (p=2 w=113) & 0.201 & 13.88 & 0.974 & 15584 & 3.70 \\
    SG: Savitzky-Golay & (p=3 w=113) & 0.196 & 13.90 & 0.974 & 14764 & 3.69 \\
    SG: Savitzky-Golay & (p=4 w=113) & 0.148 & 14.18 & 0.976 & 14732 & 3.69 \\
    Wavelet: DWT & (sym4) & 0.237 & 13.67 & 0.973 & 1130 & 5.92 \\
    Wavelet: DWT & (sym6) & 0.197 & 13.93 & 0.974 & 1229 & 5.92 \\
    Wavelet: DWT & (sym8) & \textbf{0.142} & \textbf{14.23} 

### LaTeX summary table -- single flat table, mean $\pm$ std across chips (bold = best)

In [13]:
def _summary_method_hp(label):
    """Like _split_method_label, but drops the short-form prefix ('SG: ' etc.) and
    SG's auto-tuned window (a single number isn't meaningful once averaged across
    chips, where each chip found its own optimal_w) -- matches this flat summary
    table's simpler (method name, hyperparameter) style, e.g. 'Savitzky-Golay' / '(p=2)'."""
    name, hp = _split_method_label(label)
    # 'Smoothed: Simple Moving Average' / 'SG: Savitzky-Golay' -> keep the part after
    # the colon (the long-form name); 'Wavelet: DWT' -> keep 'Wavelet' instead, since
    # that's the family name the reference table actually uses, not the transform name.
    name = 'Wavelet' if name.startswith('Wavelet:') else name.split(': ', 1)[-1]
    m = re.match(r'^\(p=(\d+) w=~?\d+\)$', hp)
    if m:
        hp = f'(p={m.group(1)})'
    return name, hp


# (metric name, unit, LaTeX arrow) -- stacked 2-line header matching the reference style.
_SUMMARY_METRIC_HEADERS = [
    ('Residual Autocorrelation', '(Lag-1)',    '\\downarrow'),
    ('SNR',                      '(dB)',       '\\uparrow'),
    ('Fidelity',                 '(Corr)',     '\\uparrow'),
    ('Speed',                    '(curves/s)', '\\uparrow'),
    ('Peak Memory',              '(MB)',       '\\downarrow'),
]


def build_denoising_summary_latex_table(metrics_by_folder, folders, caption, label,
                                        metric_cols=_LATEX_METRIC_COLS,
                                        metric_headers=_SUMMARY_METRIC_HEADERS,
                                        metric_fmt=_LATEX_METRIC_FMT,
                                        metric_direction=_LATEX_METRIC_DIRECTION):
    """Single flat table (methods as rows, metrics as columns, mean $\pm$ std across
    chips) -- unlike build_denoising_latex_table's per-chip + average panels, this is
    just the one summary panel, styled to match the target LaTeX reference exactly
    (stacked 2-line headers with a bold direction arrow, resizebox, bold-best)."""
    # show_window=True: keep the (w=~XX) suffix on SG labels so _split_method_label's
    # regex (and _summary_method_hp's window-stripping below) can actually match it --
    # show_window=False would drop it upstream, leaving nothing for either to parse.
    avg_df, std_df = _average_metrics_df(metrics_by_folder, folders, metric_cols=metric_cols,
                                         show_window=True, return_std=True)
    best_row = {
        col: (avg_df[col].idxmin() if d == 'min' else avg_df[col].idxmax())
        for col, d in zip(metric_cols, metric_direction)
    }

    header_cells = [
        f'\\begin{{tabular}}[b]{{@{{}}r@{{}}}}\\textbf{{{name}}}\\\\ '
        f'\\textbf{{{unit}}} $\\boldsymbol{{{arrow}}}$\\end{{tabular}}'
        for name, unit, arrow in metric_headers
    ]

    lines = [
        '\\begin{table}[htbp]',
        '    \\centering',
        f'    \\caption{{{caption}}}',
        f'    \\label{{{label}}}',
        '    \\small',
        '    \\resizebox{\\textwidth}{!}{',
        '    \\begin{tabular}{ll' + 'r' * len(metric_cols) + '}',
        '    \\toprule',
        '    \\textbf{Method} & \\textbf{Hyperparameter} & \n    '
        + ' & \n    '.join(header_cells) + ' \\\\',
        '    \\midrule',
    ]
    for method_label, row in avg_df.iterrows():
        name, hp = _summary_method_hp(method_label)
        cells_out = []
        for col, fmt in zip(metric_cols, metric_fmt):
            val  = fmt.format(row[col])
            sd   = std_df.loc[method_label, col] if method_label in std_df.index else np.nan
            cell = f'{val} $\\pm$ {fmt.format(sd)}' if pd.notna(sd) else val
            if method_label == best_row[col]:
                cell = f'\\textbf{{{cell}}}'
            cells_out.append(cell)
        lines.append(f'    {name} & {hp} & {" & ".join(cells_out)} \\\\')
    lines += ['    \\bottomrule', '    \\end{tabular}', '    }', '\\end{table}']
    return '\n'.join(lines)


denoising_summary_latex = build_denoising_summary_latex_table(
    denoising_metrics_by_folder, folders,
    caption='Quantitative comparison of denoising methods evaluated across all eLAMP chips. Bold marks the best result.',
    label='tab:denoising_comparison',
)
print(denoising_summary_latex)

\begin{table}[htbp]
    \centering
    \caption{Quantitative comparison of denoising methods evaluated across all eLAMP chips. Bold marks the best result.}
    \label{tab:denoising_comparison}
    \small
    \resizebox{\textwidth}{!}{
    \begin{tabular}{llrrrrr}
    \toprule
    \textbf{Method} & \textbf{Hyperparameter} & 
    \begin{tabular}[b]{@{}r@{}}\textbf{Residual Autocorrelation}\\ \textbf{(Lag-1)} $\boldsymbol{\downarrow}$\end{tabular} & 
    \begin{tabular}[b]{@{}r@{}}\textbf{SNR}\\ \textbf{(dB)} $\boldsymbol{\uparrow}$\end{tabular} & 
    \begin{tabular}[b]{@{}r@{}}\textbf{Fidelity}\\ \textbf{(Corr)} $\boldsymbol{\uparrow}$\end{tabular} & 
    \begin{tabular}[b]{@{}r@{}}\textbf{Speed}\\ \textbf{(curves/s)} $\boldsymbol{\uparrow}$\end{tabular} & 
    \begin{tabular}[b]{@{}r@{}}\textbf{Peak Memory}\\ \textbf{(MB)} $\boldsymbol{\downarrow}$\end{tabular} \\
    \midrule
    Simple Moving Average & (w=50) & 0.186 $\pm$ 0.014 & 13.64 $\pm$ 2.04 & 0.967 $\pm$ 0.017 & \textbf{416353

<>:32: SyntaxWarning: invalid escape sequence '\p'
<>:32: SyntaxWarning: invalid escape sequence '\p'
/tmp/ipykernel_2616726/2890353397.py:32: SyntaxWarning: invalid escape sequence '\p'
  """Single flat table (methods as rows, metrics as columns, mean $\pm$ std across


In [14]:
mnames = ['Residual AC lag-1', 'SNR (dB)', 'Fidelity (corr)']
mascs = [True, False, False]
top_n = 5
for mname, masc in zip(mnames, mascs):
    print(f'\nTop {top_n} methods by {mname}:')
    display(denoising_metrics.sort_values(by=mname, ascending=masc)[[mname]].head(top_n))


Top 5 methods by Residual AC lag-1:


,Residual AC lag-1
Wavelet (sym8),0.154239
SG p=4 (w=109),0.158827
SG p=3 (w=109),0.200005
Smoothed (w=50),0.204742
Wavelet (sym6),0.206066



Top 5 methods by SNR (dB):


,SNR (dB)
Wavelet (sym8),13.317272
SG p=4 (w=109),13.266785
Wavelet (sym6),13.020261
SG p=3 (w=109),13.015920
Wavelet (sym4),13.005883



Top 5 methods by Fidelity (corr):


,Fidelity (corr)
Wavelet (sym8),0.972904
SG p=4 (w=109),0.972616
SG p=3 (w=109),0.971169
Wavelet (sym6),0.971146
Wavelet (sym4),0.971078
